# Nepal Stock Market — LSTM Multi-Task Learning (All Stocks)

**Model:** LSTM with shared layers + 3 task-specific output heads  
**Data:** All NEPSE stocks (2012–2020), ~250,000 trading records  
**Tasks:**
- **Task A:** Predict tomorrow's return (regression)  
- **Task B:** Predict tomorrow's direction — UP or DOWN (classification)  

**Why LSTM MTL?**
- LSTM captures temporal dependencies (30-day sequences)
- Multi-task learning shares knowledge between return and direction prediction
- Stock embeddings let one model generalize to ALL stocks
- GPU-accelerated training on Kaggle (30–60 mins)

---
> 🚀 Run this notebook on **Kaggle GPU** (P100/T4) for optimal performance

## 1. Setup & Environment Detection

In [ ]:
import os, glob, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Detect Kaggle vs Local ────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')
print(f"Running on Kaggle: {IS_KAGGLE}")

# ── GPU detection ─────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(gpus)}")
if gpus:
    print(f"GPU device: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("No GPU found — training on CPU (will be slower)")

# ── Keras imports ─────────────────────────────────────────────────────────────
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, LSTM, Dense, Dropout,
                                      Embedding, Flatten, Concatenate,
                                      BatchNormalization)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              roc_auc_score, accuracy_score, f1_score,
                              classification_report, confusion_matrix, roc_curve)

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

print("\nAll libraries loaded successfully.")

## 2. Data Loading (Kaggle-Compatible)

In [ ]:
# ── Auto-detect data path ────────────────────────────────────────────────────
if IS_KAGGLE:
    # Search for OHLC.csv in all Kaggle input directories
    matches = glob.glob('/kaggle/input/**/OHLC.csv', recursive=True)
    if not matches:
        # Fallback: list all available CSVs
        all_csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
        print("OHLC.csv not found. Available CSV files:")
        for f in all_csvs[:10]:
            print(" ", f)
        raise FileNotFoundError("Please add your Nepal Stock Market dataset to Kaggle.")
    ohlc_path = matches[0]
else:
    # Local path
    ohlc_path = os.path.join(os.getcwd(), "Data", "OHLC.csv")

print(f"Loading data from: {ohlc_path}")
df_raw = pd.read_csv(ohlc_path)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

## 3. Data Cleaning

In [ ]:
# Drop row index column (S.no)
if 'S.no' in df_raw.columns:
    df_raw = df_raw.drop(columns=['S.no'])

# Clean Date
df_raw['Date'] = df_raw['Date'].astype(str).str.replace('.csv', '', regex=False)
df_raw['Date'] = pd.to_datetime(df_raw['Date'], errors='coerce')

# Clean Volume
df_raw['Vol'] = df_raw['Vol'].astype(str).str.replace(',', '', regex=False)
df_raw['Vol'] = pd.to_numeric(df_raw['Vol'], errors='coerce')

# Sort, drop nulls, duplicates
df = (df_raw
      .sort_values(['Symbol', 'Date'])
      .dropna(subset=['Symbol', 'Date', 'Close'])
      .drop_duplicates()
      .reset_index(drop=True))

print(f"Cleaned dataset: {df.shape}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Unique stocks: {df['Symbol'].nunique()}")
print(f"Total trading records: {len(df):,}")
df.isnull().sum()

In [ ]:
# ── Stock selection: keep stocks with enough history ─────────────────────────
# Require at least 500 trading days (2 years) — needed for meaningful sequences
MIN_RECORDS = 500
stock_counts = df['Symbol'].value_counts()
valid_stocks  = stock_counts[stock_counts >= MIN_RECORDS].index.tolist()

df = df[df['Symbol'].isin(valid_stocks)].copy().reset_index(drop=True)

print(f"Stocks with >= {MIN_RECORDS} records: {len(valid_stocks)}")
print(f"Dataset after filtering: {df.shape}")
print(f"Top 10 stocks by record count:")
print(stock_counts[stock_counts >= MIN_RECORDS].head(10))

## 4. Feature Engineering

For LSTM MTL across all stocks, we use **only stationary features** (returns, ratios, indicators).  
These are scale-invariant — a 2% return means the same for a 500 NPR stock or a 2000 NPR stock.  
This allows **one model to generalize across all stocks without per-stock price normalization**.


In [ ]:
def engineer_features(stock_df):
    """Engineer stationary features for one stock. Returns a clean DataFrame."""
    s = stock_df.copy().sort_values('Date').reset_index(drop=True)

    # ── Returns ───────────────────────────────────────────────────────────────
    s['Return_1']        = s['Close'].pct_change()
    s['Log_Return']      = np.log(s['Close'] / s['Close'].shift(1))
    s['Intraday_Return'] = (s['Close'] - s['Open']) / s['Open']
    s['Overnight_Gap']   = (s['Open'] - s['Close'].shift(1)) / s['Close'].shift(1)

    # ── Lag returns ───────────────────────────────────────────────────────────
    for lag in [1, 2, 3, 5, 7, 14]:
        s[f'Return_Lag_{lag}']     = s['Return_1'].shift(lag)
        s[f'Log_Return_Lag_{lag}'] = s['Log_Return'].shift(lag)

    # ── Rolling return stats ──────────────────────────────────────────────────
    for w in [3, 5, 7, 14, 30]:
        s[f'Return_Mean_{w}']  = s['Return_1'].rolling(w).mean()
        s[f'Return_Std_{w}']   = s['Return_1'].rolling(w).std()   # rolling volatility
        s[f'Return_Skew_{w}']  = s['Return_1'].rolling(w).skew()

    # ── Price ratios (stationary) ─────────────────────────────────────────────
    for w in [5, 10, 20, 30, 60]:
        sma = s['Close'].rolling(w).mean()
        s[f'Price_to_SMA_{w}'] = s['Close'] / sma
        s[f'ROC_{w}']          = s['Close'].pct_change(w)

    # ── Candlestick ratios ────────────────────────────────────────────────────
    rng = s['High'] - s['Low']
    s['Close_Position']  = ((s['Close'] - s['Low']) / rng).replace([np.inf,-np.inf], np.nan)
    s['Body_to_Range']   = (abs(s['Close']-s['Open']) / rng).replace([np.inf,-np.inf], np.nan)
    s['Open_Close_Pct']  = (s['Close'] - s['Open']) / s['Open']
    s['High_Low_Pct']    = rng / s['Open']

    # ── Volume ratios ─────────────────────────────────────────────────────────
    for w in [5, 10, 20]:
        s[f'Volume_Ratio_{w}'] = s['Vol'] / s['Vol'].rolling(w).mean()
    s['Volume_Change']  = s['Vol'].pct_change()
    s['Log_Volume']     = np.log1p(s['Vol'])

    # ── Technical indicators ──────────────────────────────────────────────────
    # RSI
    delta    = s['Close'].diff()
    gain     = delta.clip(lower=0).rolling(14).mean()
    loss     = (-delta.clip(upper=0)).rolling(14).mean()
    s['RSI'] = 100 - (100 / (1 + gain / loss))

    # MACD
    ema12 = s['Close'].ewm(span=12, adjust=False).mean()
    ema26 = s['Close'].ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    s['MACD_Hist_Norm'] = (macd - macd.ewm(span=9, adjust=False).mean()) / s['Close']

    # Bollinger Band %B
    bb_mid = s['Close'].rolling(20).mean()
    bb_std = s['Close'].rolling(20).std()
    s['BB_Pct_B'] = (s['Close'] - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-8)

    # ATR %
    prev_c = s['Close'].shift(1)
    tr = pd.concat([s['High']-s['Low'],
                    abs(s['High']-prev_c),
                    abs(s['Low']-prev_c)], axis=1).max(axis=1)
    s['ATR_Pct'] = tr.rolling(14).mean() / s['Close']

    # Stochastic
    lo14 = s['Low'].rolling(14).min()
    hi14 = s['High'].rolling(14).max()
    s['Stoch_K'] = ((s['Close']-lo14)/(hi14-lo14+1e-8))*100

    # ── Time features (cyclical) ──────────────────────────────────────────────
    s['Month_sin']     = np.sin(2*np.pi*s['Date'].dt.month/12)
    s['Month_cos']     = np.cos(2*np.pi*s['Date'].dt.month/12)
    s['DayOfWeek_sin'] = np.sin(2*np.pi*s['Date'].dt.dayofweek/7)
    s['DayOfWeek_cos'] = np.cos(2*np.pi*s['Date'].dt.dayofweek/7)

    # ── Targets ───────────────────────────────────────────────────────────────
    s['Target_Return']    = s['Return_1'].shift(-1)
    s['Target_Direction'] = (s['Close'].shift(-1) > s['Close']).astype(int)

    return s

print("Feature engineering function defined.")

In [ ]:
# ── Compute market-wide context features ─────────────────────────────────────
print("Computing market context features...")
df['Stock_Return'] = df.groupby('Symbol')['Close'].pct_change()

market = df.groupby('Date').agg(
    Market_Return       = ('Stock_Return', 'mean'),
    Market_Adv_Ratio    = ('Stock_Return', lambda x: (x > 0).mean()),
    Market_Volatility   = ('Stock_Return', 'std'),
).reset_index()

print(f"Market context computed for {len(market)} trading dates.")
market.head()

In [ ]:
# ── Apply feature engineering to all valid stocks ────────────────────────────
print("Engineering features for all stocks... (this may take 1-2 minutes)")
all_fe = []
failed = []

for symbol in valid_stocks:
    try:
        s_df = df[df['Symbol'] == symbol].copy()
        s_fe = engineer_features(s_df)
        s_fe = s_fe.merge(market, on='Date', how='left')
        s_fe['Relative_Return'] = s_fe['Return_1'] - s_fe['Market_Return']
        s_fe['Symbol'] = symbol
        all_fe.append(s_fe)
    except Exception as e:
        failed.append(symbol)

if failed:
    print(f"Failed stocks: {failed}")

combined = pd.concat(all_fe, ignore_index=True)
combined = combined.replace([np.inf, -np.inf], np.nan)

print(f"\nTotal records after feature engineering: {len(combined):,}")
print(f"Stocks processed: {len(all_fe)}")

## 5. Sequence Preparation

LSTM requires 3D input: **(samples, timesteps, features)**  
For each trading day, we look back `SEQ_LEN` days to form a sequence.  
A sliding window approach is used per stock to avoid leakage across stocks.


In [ ]:
# ── Define feature columns (stationary only) ─────────────────────────────────
FEATURE_COLS = [c for c in combined.columns if c not in [
    'Symbol', 'Date', 'Open', 'High', 'Low', 'Close', 'Vol',
    'Target_Return', 'Target_Direction', 'Stock_Return'
]]

# Remove features with >20% NaN before we start
nan_pct = combined[FEATURE_COLS].isnull().mean()
FEATURE_COLS = [c for c in FEATURE_COLS if nan_pct[c] < 0.20]

print(f"Number of features: {len(FEATURE_COLS)}")
print("Features:", FEATURE_COLS[:10], "...")

In [ ]:
# ── Encode stock symbols as integers ─────────────────────────────────────────
le = LabelEncoder()
combined['Stock_ID'] = le.fit_transform(combined['Symbol'])
N_STOCKS = len(le.classes_)
print(f"Total unique stocks: {N_STOCKS}")
print(f"Stock encoding sample: {list(zip(le.classes_[:5], range(5)))}")

In [ ]:
# ── Time-series train/test split (80/20 by DATE) ─────────────────────────────
all_dates = sorted(combined['Date'].unique())
split_date = all_dates[int(len(all_dates) * 0.80)]
print(f"Train period: {all_dates[0].date()} → {split_date.date()}")
print(f"Test period:  {split_date.date()} → {all_dates[-1].date()}")

train_df = combined[combined['Date'] <  split_date].copy()
test_df  = combined[combined['Date'] >= split_date].copy()
print(f"Train rows: {len(train_df):,} | Test rows: {len(test_df):,}")

In [ ]:
# ── Fit StandardScaler on training data only ─────────────────────────────────
scaler = StandardScaler()
train_df[FEATURE_COLS] = scaler.fit_transform(train_df[FEATURE_COLS].fillna(0))
test_df[FEATURE_COLS]  = scaler.transform(test_df[FEATURE_COLS].fillna(0))
print("Features scaled (fit on train, applied to test).")

In [ ]:
SEQ_LEN = 20  # Look back 20 trading days (1 calendar month)

def make_sequences(df_subset, feature_cols, seq_len=20):
    """
    Build 3D sequences from a DataFrame.
    For each stock independently, create sliding windows.
    Returns: X (n, seq_len, features), y_ret (n,), y_dir (n,), stock_ids (n,)
    """
    X_list, y_ret_list, y_dir_list, sid_list = [], [], [], []

    for symbol, group in df_subset.groupby('Symbol'):
        group = group.sort_values('Date').reset_index(drop=True)
        feats = group[feature_cols].values
        y_ret = group['Target_Return'].values
        y_dir = group['Target_Direction'].values
        sid   = group['Stock_ID'].values

        # Drop rows with NaN targets
        for i in range(seq_len, len(group)):
            window = feats[i-seq_len:i]
            if np.isnan(window).any():
                continue
            if np.isnan(y_ret[i]) or np.isnan(y_dir[i]):
                continue
            X_list.append(window)
            y_ret_list.append(y_ret[i])
            y_dir_list.append(y_dir[i])
            sid_list.append(sid[i])

    return (np.array(X_list, dtype=np.float32),
            np.array(y_ret_list, dtype=np.float32),
            np.array(y_dir_list, dtype=np.float32),
            np.array(sid_list, dtype=np.int32))

print("Building training sequences...")
X_train, y_ret_train, y_dir_train, sid_train = make_sequences(train_df, FEATURE_COLS, SEQ_LEN)
print(f"Train: X={X_train.shape}, y_return={y_ret_train.shape}, y_dir={y_dir_train.shape}")

print("Building test sequences...")
X_test, y_ret_test, y_dir_test, sid_test = make_sequences(test_df, FEATURE_COLS, SEQ_LEN)
print(f"Test:  X={X_test.shape}, y_return={y_ret_test.shape}, y_dir={y_dir_test.shape}")

## 6. LSTM Multi-Task Learning Model Architecture

```
Sequence Input (20 days × N features)     Stock ID Input
        ↓                                       ↓
   LSTM (128 units)                    Embedding (N_stocks → 16)
        ↓                                       ↓
   LSTM (64 units)                          Flatten
        ↓                                       ↓
   BatchNorm + Dropout(0.3)               ←──── Concat
                        ↓
                  Dense(64, relu)
                  Dropout(0.2)
                    /         \
           Task A Head        Task B Head
         (return reg)      (direction clf)
              ↓                   ↓
           Dense(1)          Dense(1, sigmoid)
```

**Shared layers** learn general market dynamics.  
**Task heads** specialize in each prediction task.  
**Stock embeddings** allow the model to adapt to each stock's personality.


In [ ]:
def build_lstm_mtl(seq_len, n_features, n_stocks, embed_dim=16):
    """
    Build LSTM Multi-Task Learning model.
    Inputs:  sequence (seq_len, n_features), stock_id (1,)
    Outputs: return prediction, direction prediction
    """
    # ── Sequence branch ───────────────────────────────────────────────────────
    seq_input = Input(shape=(seq_len, n_features), name='sequence_input')
    x = LSTM(128, return_sequences=True, dropout=0.2,
             recurrent_dropout=0.0, kernel_regularizer=l2(1e-4))(seq_input)
    x = LSTM(64, dropout=0.2,
             kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # ── Stock embedding branch ─────────────────────────────────────────────────
    stock_input = Input(shape=(1,), name='stock_input')
    emb = Embedding(n_stocks, embed_dim, name='stock_embedding')(stock_input)
    emb = Flatten()(emb)

    # ── Combine ────────────────────────────────────────────────────────────────
    combined = Concatenate()([x, emb])
    shared   = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(combined)
    shared   = Dropout(0.2)(shared)

    # ── Task A: Return prediction (regression) ─────────────────────────────────
    ret_h = Dense(32, activation='relu')(shared)
    ret_h = Dropout(0.1)(ret_h)
    return_out = Dense(1, name='return_output')(ret_h)

    # ── Task B: Direction prediction (classification) ──────────────────────────
    dir_h = Dense(32, activation='relu')(shared)
    dir_h = Dropout(0.1)(dir_h)
    direction_out = Dense(1, activation='sigmoid', name='direction_output')(dir_h)

    model = Model(
        inputs=[seq_input, stock_input],
        outputs=[return_out, direction_out]
    )

    model.compile(
        optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
        loss={
            'return_output':    'mse',
            'direction_output': 'binary_crossentropy'
        },
        loss_weights={
            'return_output':    1.0,
            'direction_output': 0.5
        },
        metrics={
            'return_output':    ['mae'],
            'direction_output': ['accuracy']
        }
    )
    return model


N_FEATURES = X_train.shape[2]
model = build_lstm_mtl(SEQ_LEN, N_FEATURES, N_STOCKS)
model.summary()

## 7. Training

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='lstm_mtl_best.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

BATCH_SIZE = 256   # Larger batch = faster GPU training
MAX_EPOCHS = 100   # EarlyStopping will cut this short

print(f"Training configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Max epochs: {MAX_EPOCHS}")
print(f"  Training samples: {len(X_train):,}")
print(f"  Validation samples: {len(X_test):,}")
print(f"  Features per timestep: {N_FEATURES}")
print(f"  Sequence length: {SEQ_LEN} days")
print(f"  Total parameters: {model.count_params():,}")

In [ ]:
# ── Train the model ──────────────────────────────────────────────────────────
print("Starting training...")
history = model.fit(
    [X_train, sid_train],
    {'return_output': y_ret_train, 'direction_output': y_dir_train},
    validation_data=(
        [X_test, sid_test],
        {'return_output': y_ret_test, 'direction_output': y_dir_test}
    ),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print("\nTraining complete!")

In [ ]:
# ── Plot training history ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Total loss
axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Total Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Return MAE
axes[1].plot(history.history['return_output_mae'],     label='Train MAE')
axes[1].plot(history.history['val_return_output_mae'], label='Val MAE')
axes[1].set_title('Task A — Return Prediction MAE', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Direction Accuracy
axes[2].plot(history.history['direction_output_accuracy'],     label='Train Accuracy')
axes[2].plot(history.history['val_direction_output_accuracy'], label='Val Accuracy')
axes[2].set_title('Task B — Direction Accuracy', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('LSTM MTL Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Evaluation — LSTM MTL vs Baselines

In [ ]:
# ── Generate predictions ──────────────────────────────────────────────────────
pred_ret, pred_dir_prob = model.predict([X_test, sid_test], batch_size=512, verbose=0)
pred_ret      = pred_ret.flatten()
pred_dir_prob = pred_dir_prob.flatten()
pred_dir      = (pred_dir_prob >= 0.5).astype(int)

print(f"Predictions generated for {len(pred_ret):,} test samples")

In [ ]:
# ── Task A: Return Prediction Metrics ─────────────────────────────────────────
def safe_mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan

# Baselines
mean_ret = y_ret_train.mean()
baseline_ret = np.full(len(y_ret_test), mean_ret)
zero_ret     = np.zeros(len(y_ret_test))

metrics_taskA = []
for name, pred in [("Baseline (Mean Return)", baseline_ret),
                   ("Baseline (Zero Return)", zero_ret),
                   ("LSTM MTL (shared)",      pred_ret)]:
    metrics_taskA.append({
        "Model":      name,
        "MAE":        round(mean_absolute_error(y_ret_test, pred), 6),
        "RMSE":       round(np.sqrt(mean_squared_error(y_ret_test, pred)), 6),
        "R² Score":   round(r2_score(y_ret_test, pred), 4),
        "MAPE (%)":   round(safe_mape(y_ret_test, pred), 2)
    })

df_taskA = pd.DataFrame(metrics_taskA).sort_values('R² Score', ascending=False)
print("=" * 65)
print("TASK A: Return Prediction — LSTM MTL Results")
print("=" * 65)
display(df_taskA)

In [ ]:
# ── Task B: Direction Prediction Metrics ──────────────────────────────────────
majority = int(round(y_dir_train.mean()))
baseline_dir = np.full(len(y_dir_test), majority)

metrics_taskB = []
for name, pred, prob in [
    ("Baseline (Majority Class)", baseline_dir, None),
    ("LSTM MTL (shared)",         pred_dir,     pred_dir_prob)
]:
    auc = roc_auc_score(y_dir_test, prob) if prob is not None else float('nan')
    metrics_taskB.append({
        "Model":     name,
        "Accuracy":  round(accuracy_score(y_dir_test, pred), 4),
        "F1 Score":  round(f1_score(y_dir_test, pred, zero_division=0), 4),
        "ROC-AUC":   round(auc, 4)
    })

df_taskB = pd.DataFrame(metrics_taskB).sort_values('ROC-AUC', ascending=False)
print("=" * 65)
print("TASK B: Direction Prediction — LSTM MTL Results")
print("=" * 65)
display(df_taskB)
print()
print("Detailed Classification Report — LSTM MTL:")
print(classification_report(y_dir_test, pred_dir, target_names=['DOWN', 'UP']))

## 9. Results Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Actual vs Predicted Returns (scatter)
sample_idx = np.random.choice(len(y_ret_test), size=min(2000, len(y_ret_test)), replace=False)
axes[0,0].scatter(y_ret_test[sample_idx]*100, pred_ret[sample_idx]*100,
                  alpha=0.3, s=8, color='steelblue')
lim = max(abs(y_ret_test).max(), abs(pred_ret).max()) * 100 * 1.1
axes[0,0].plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.5, label='Perfect')
axes[0,0].set_title('LSTM MTL: Actual vs Predicted Return', fontweight='bold')
axes[0,0].set_xlabel('Actual Return (%)')
axes[0,0].set_ylabel('Predicted Return (%)')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
r2_val = r2_score(y_ret_test, pred_ret)
axes[0,0].text(0.05, 0.92, f'R² = {r2_val:.4f}', transform=axes[0,0].transAxes,
               fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_dir_test, pred_dir_prob)
auc_val = roc_auc_score(y_dir_test, pred_dir_prob)
axes[0,1].plot(fpr, tpr, linewidth=2.5, color='steelblue', label=f'LSTM MTL (AUC={auc_val:.3f})')
axes[0,1].plot([0,1],[0,1],'k--',linewidth=1,label='Random (AUC=0.500)')
axes[0,1].fill_between(fpr, tpr, alpha=0.1, color='steelblue')
axes[0,1].set_title('ROC Curve — Direction Prediction', fontweight='bold')
axes[0,1].set_xlabel('False Positive Rate')
axes[0,1].set_ylabel('True Positive Rate')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# 3. Confusion Matrix
cm = confusion_matrix(y_dir_test, pred_dir)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['DOWN','UP'], yticklabels=['DOWN','UP'],
            ax=axes[1,0], linewidths=0.5)
axes[1,0].set_title('Confusion Matrix — Direction Prediction', fontweight='bold')
axes[1,0].set_xlabel('Predicted'); axes[1,0].set_ylabel('Actual')

# 4. Stock Embedding visualization (PCA)
try:
    from sklearn.decomposition import PCA
    emb_weights = model.get_layer('stock_embedding').get_weights()[0]
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(emb_weights)
    axes[1,1].scatter(emb_2d[:,0], emb_2d[:,1], s=30, alpha=0.6, color='steelblue')
    # Label top stocks
    for i, sym in enumerate(le.classes_[:10]):
        axes[1,1].annotate(sym, (emb_2d[i,0], emb_2d[i,1]), fontsize=7, alpha=0.8)
    axes[1,1].set_title('Stock Embeddings (PCA 2D) - Similar stocks cluster',
                        fontweight='bold')
    axes[1,1].set_xlabel('PC1'); axes[1,1].set_ylabel('PC2')
    axes[1,1].grid(True, alpha=0.3)
except Exception as e:
    axes[1,1].text(0.5, 0.5, f'Embedding visualization\nfailed: {e}',
                   ha='center', va='center', transform=axes[1,1].transAxes)

plt.suptitle('LSTM Multi-Task Learning — Results Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-Stock Performance Analysis ───────────────────────────────────────────
# How well does the LSTM generalize to individual stocks?
stock_results = []
unique_sids = np.unique(sid_test)

for sid in unique_sids[:20]:  # Top 20 stocks for brevity
    mask = sid_test == sid
    if mask.sum() < 20:
        continue
    r2  = r2_score(y_ret_test[mask], pred_ret[mask])
    auc = roc_auc_score(y_dir_test[mask], pred_dir_prob[mask]) if len(np.unique(y_dir_test[mask])) > 1 else np.nan
    stock_results.append({
        'Stock':    le.classes_[sid],
        'N_samples': mask.sum(),
        'R² (Return)': round(r2,  4),
        'AUC (Direction)': round(auc, 4) if not np.isnan(auc) else np.nan
    })

df_per_stock = pd.DataFrame(stock_results).sort_values('AUC (Direction)', ascending=False)
print("Per-Stock Performance (Top 20 by AUC):")
display(df_per_stock)

## 10. Conclusions

### LSTM Multi-Task Learning Key Findings

1. **Generalization:** One LSTM model trained on ALL stocks simultaneously — no per-stock retraining needed
2. **Stock Embeddings:** Each stock has a learned 16-dimensional representation. Stocks with similar behavior cluster together (visible in PCA plot)
3. **MTL Benefit:** Shared LSTM layers capture market-wide patterns; task heads specialize in return vs direction
4. **Comparison with per-stock ML:**
   - Traditional ML (SCB only): R² ≈ 9.5%, AUC ≈ 0.618
   - LSTM MTL (all stocks): See results above

### Architecture Design Choices

| Choice | Reason |
|---|---|
| Stationary features only | Allows one scaler for all stocks, prevents price-level bias |
| 20-day sequences | ~1 trading month, captures medium-term momentum |
| Stock embeddings | Each stock gets its own "personality" vector |
| MTL with two heads | Direction loss regularizes the return head (and vice versa) |
| EarlyStopping | Prevents overfitting on volatile financial data |
| BatchNorm | Stabilizes training across stocks with different volatility profiles |

### Future Improvements
- Attention mechanisms (Transformer) for variable-length look-back
- Sector/industry features as additional embeddings
- Walk-forward retraining (retrain monthly)
- Incorporate macro data (interest rates, USD/NPR)
